In [1]:
import ROOT as rt
import uproot
import numpy as np
from collections import OrderedDict
import os
import sys
from array import array
sys.path.append(os.getcwd().replace('scripts', 'lib'))

import tdrstyle
a = tdrstyle.setTDRStyle()

print(sys.version)

Welcome to JupyROOT 6.30/02
3.9.12 (main, Jul 11 2023, 14:37:47) 
[GCC 11.3.0]


# load signal ntuples

Load signal ntuples with 4 different LLP lifetimes (0.1, 1, 10, 100m) that contain generator-level LLP information

In [2]:
fpath =OrderedDict()
tree = OrderedDict()

mass = [40]

OLD_CTAU = np.array([100, 1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for ct in OLD_CTAU:
    key = 'MC_40_'+str(ct)                       
    fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'

    
NEvents = {}
NEvents_genweight = {}
for k,v in fpath.items():
    root_dir = uproot.open(v) 
    if not root_dir: 
        print(k, "zombie")
        continue
    tree[k] = root_dir['MuonSystem']


# Ex0 - calculate the LLP proper decay length from the LLP four momentum

Here you will calculate the LLP proper decay length by converting from the lab frame decay length to the decay length in the LLP's rest frame, with the following equation:

$d = \beta\gamma\tau$, 

where d is the lab frame decay length, $\beta$ is the LLP velocity, $\gamma$ is the LLP Lorentz factor, and $\tau$ is the LLP proper decay lifetime.

You will use the following branches in the ROOT tree 
* `gLLP_decay_vertex_x`, in units of cm
* `gLLP_decay_vertex_y`, in units of cm
* `gLLP_decay_vertex_z`, in units of cm
* `gLLP_beta` , ranges from 0 to 1

and you can access a branch by doing `T['branch name'].array()` below.

For example to access `gLLP_beta` with `T['gLLP_beta'].array()`. 

The shape of the LLP-level branches is (number of LLPs, 2), because there are two LLPs per event.

Fill in the righthand side of each equation below with the 4 branches mentioned to calculate the proper decay length

In [ ]:
gLLP_ctau = {}
for k, T in tree.items():
    # lab-frame decay length: distance from IP (0,0,0) to decay vertex, in cm
    gLLP_decay_vertex = np.sqrt(T['gLLP_decay_vertex_x'].array()**2 + T['gLLP_decay_vertex_y'].array()**2 + T['gLLP_decay_vertex_z'].array()**2)
    gLLP_beta = T['gLLP_beta'].array()
    # gamma = 1/sqrt(1-beta^2)
    gLLP_gamma = 1.0/np.sqrt(1.0 - gLLP_beta**2)
    # proper decay length: ctau = d/(beta*gamma), in cm
    gLLP_ctau[k] = gLLP_decay_vertex/(gLLP_gamma*gLLP_beta)

### Ex0 explanation

The tree only stores the **lab frame** decay vertex, but the lifetime of a particle is defined in its **own rest frame**. The two are related by time dilation: a particle that lives a proper time $\tau$ travels the lab-frame distance $d = \beta\gamma c\tau$, so

$c\tau = \dfrac{d}{\beta\gamma}$

Step by step:
1. `gLLP_decay_vertex` = $\sqrt{x^2+y^2+z^2}$ is the distance from the interaction point (0,0,0) to the decay vertex, i.e. the lab-frame decay length $d$ in cm.
2. `gLLP_beta` is $\beta = v/c$ (0 to 1) and `gLLP_gamma` = $1/\sqrt{1-\beta^2}$ is the Lorentz boost.
3. Dividing $d$ by $\beta\gamma$ de-boosts the measured displacement back into the LLP rest frame, giving the **proper decay length** $c\tau$ (still in cm).

Because the decay is a Poisson process, the distribution of $c\tau$ values must be exponential, with a mean equal to the $c\tau$ that was set at generation time. Note that the $c\tau$ we get out is a Lorentz invariant property of the particle, while $d$ is not - it depends on how boosted the LLP happens to be event by event. A common pitfall: the branches are in **cm** while the file names (e.g. `ctau-1000`) are in **mm**.

# Ex1 - plot the LLP proper decay length

In this exercise, you will plot the `gLLP_ctau` array for 1m and 10m sample.

You will then perform an exponential fit to verify that the shape is exponential and linear in logY scale.

From the exponential fit, you can extract the mean proper decay length of the LLP and verify that it agrees with what you expect from the sample (1m or 10m)

You have to replace the `REPLACE_ME` variable below to fill in the correct variable to extract the fitted ctau by using the variable `fitted_slope`. 

Note that the unit in the tree branches are in cm, while the unit in the ROOT file names are in mm.

In [ ]:
%%time

Nevents_todraw = 10000
c = rt.TCanvas('c','c', 800, 600)
h = {}

for i, k in enumerate([ 'MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 10000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')
    for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    h[k].Scale(1./h[k].Integral())  #normalize histogram to 1
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')

    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10./fit.GetParameter(1)))
    #######################################
    
c.SetLogy()
c.Draw()


### Ex1 explanation

We fill the proper decay length of the *first* LLP of each event (`gLLP_ctau[k][j][0]`, in cm) into a normalized histogram and look at it on a log-y axis. A pure exponential N(t) = exp(-t/tau) becomes a straight line in log space,

ln N = p0 + p1 * t,  with  p1 = -1/tau,

which is exactly what ROOT's built-in `expo` function fits, so both samples should look linear.

To get the fitted ctau back out of the slope we invert that relation: ctau = -1/p1. The fitted variable (the histogram axis) is in **cm**, so -1/p1 is in cm, while the print statement asks for **mm**, hence

`-10./fit.GetParameter(1)`

(`fit.GetParameter(1)` returns p1, the slope). Running the cell gives about 1000 mm for `MC_40_1000` and 10000 mm for `MC_40_10000`, i.e. exactly the ctau written in the file names. That confirms (a) the proper decay length really is exponential and (b) the unit bookkeeping is right. Forgetting the factor of 10 (mm in the file name vs. cm in the branches) is the classic pitfall - the fit would then look 10x off.

# Ex2 - reweight LLP lifetime 

In this exercise you will learn how to reweight the LLP lifetime.

In these signal events, there are two LLPs and the decay position of the two LLPs in each event are independent, and each LLP decays with an exponential probability, the distribution of events is simply the product of the two LLP decay probabilities:
\begin{equation}
  p(t_1, t_2 | \tau) = \frac{1}{\tau^2}\exp^{-t_1/\tau}\exp^{-t_2/\tau}
\end{equation}
where $\tau$ is the mean proper decay length of the LLPs, and $t_1$ and $t_2$ are the lifetimes of the first LLP and second LLP in their own rest frame respectively, which are the `gLLP_ctau` variable that you have calculated in Ex1.

To obtain a sample with lifetime $\tau_{new}$ from a sample with lifetime $\tau_{old}$, we
assign a weight, which is the ratio between equation above with parameter $\tau_{new}$
and:
\begin{equation}
  w = (\frac{\tau_{old}}{\tau_{new}})^2 \exp[(t_1+t_2)\times(\frac{1}{\tau_{old}}-\frac{1}{\tau_{new}})]
\end{equation}

You will implement this equation in the function `weight_calc` below

In [5]:
'''
arguements: 
llp_ctau is the list of sum of proper decay length, which is t1+t2 in the equation
new_ctau is tau_new in the equation
old_ctau is tau_old in the equation
All units in cm

return event weight 

'''


def weight_calc(llp_ct, new_ctau, old_ctau):
    denominator = np.exp(-1.0*llp_ct/old_ctau)/old_ctau**2
    numerator = np.exp(-1.0*llp_ct/new_ctau)/new_ctau**2
    weight = numerator/denominator
    
    return weight

### Ex2 (`weight_calc`) explanation

Each event has two independent LLPs, so the probability density of measuring proper decay lengths $t_1, t_2$ for a hypothesis $\tau$ is the product of two exponentials

$p(t_1,t_2|\tau) = \frac{1}{\tau^2}\,e^{-t_1/\tau}\,e^{-t_2/\tau} = \frac{1}{\tau^2} e^{-(t_1+t_2)/\tau}.$

Because only the **sum** $t_1+t_2$ (`llp_ct`) enters, we can reweight an event by the ratio of the two probability densities:

$w = \frac{p(t_1+t_2|\tau_{new})}{p(t_1+t_2|\tau_{old})} = \left(\frac{\tau_{old}}{\tau_{new}}\right)^2 \exp\left[(t_1+t_2)\left(\frac{1}{\tau_{old}}-\frac{1}{\tau_{new}}\right)\right]$

In the code the numerator is literally the `new` pdf and the denominator (already given) is the `old` pdf:

~~~
numerator = np.exp(-1.0*llp_ct/new_ctau)/new_ctau**2
weight = numerator/denominator
~~~

so the $(\tau_{old}/\tau_{new})^2$ prefactor appears automatically from the $1/\tau^2$ normalizations of the two LLPs. This is the same implementation as the reference `weight_calc` in `lib/helper.py`. All arguments must be in the **same units** (cm here): the function knows nothing about mm, so passing mm for one argument and cm for another silently gives a wrong (but perfectly smooth-looking) answer.

## reweight from 10m to 1m

In this exercise you will 
 - use the reweighting function that you have just implemented to reweight the 10m sample to 1m 
 - plot it against the original 1m sample to validate the reweighting function.

You will need to fill in the arguments for the `weight_calc()` function with `llp_ct` and the two lifetimes. 

Be sure to convert everything to cm.

You will then perform an exponential fit to verify that the extracted mean proper decay length of the LLP is 1m for both by replacing the variable `REPLACE_ME`

In [ ]:
%%time
c = rt.TCanvas('c','c', 800, 600)
h = {}
Nevents_todraw = 10000

for i, k in enumerate(['MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 1000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')
    
    if k == 'MC_40_1000': 
        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    else: 
        llp_ct = np.sum(gLLP_ctau[k], axis = 1)
        
        # Fill in the arguments in weight_calc()
        # reweight the 10 m sample (old_ctau = 1000 cm) down to 1 m (new_ctau = 100 cm)
        weight = weight_calc(llp_ct, 100.0, 1000.0)
                
        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0], weight[j])
    
    h[k].Scale(1./h[k].Integral())
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')
    

    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10./fit.GetParameter(1)))

    #######################################
    
c.SetLogy()
c.Draw()


### Ex2 (reweighting 10 m to 1 m) explanation

`llp_ct = np.sum(gLLP_ctau[k], axis=1)` adds up the proper decay lengths of the two LLPs in each event, which is exactly the $t_1+t_2$ that `weight_calc` expects. Everything has to be passed in **cm**:

~~~
weight = weight_calc(llp_ct, 100.0, 1000.0)
~~~

i.e. `new_ctau = 100 cm = 1 m` and `old_ctau = 1000 cm = 10 m`, the ctau written in the two file names (`ctau-1000`/`ctau-10000` are in mm). The weights are then used as the second argument of `h[k].Fill(value, weight)`, so the 10 m histogram is rebuilt as if it had been generated with a 1 m lifetime.

Validation: the two curves on the log-y plot should lie on top of each other and both exponential fits should print a fitted ctau of about 1000 mm. Notice that the weights are smaller than 1 for most events (the 10 m sample has too many late decays) and are largest for the small-$t$ events, which are the ones the 1 m sample needs. That is why we reweight from a *long* lifetime to a *short* one.

### Discussion 2.2 - which direction should we reweight?

We always reweight from a **larger** ctau to a **smaller** one. With $\tau_{new} < \tau_{old}$ the exponent $(t_1+t_2)(1/\tau_{old}-1/\tau_{new})$ is **negative**, so the weights only ever shrink the long-decay tail: $w \le (\tau_{old}/\tau_{new})^2$ and, more importantly, the events that carry the most weight are the short-decay ones, which the long-lived sample is full of. The weighted histogram is therefore statistically well behaved - it is just as good as a sample generated with $\tau_{new}$.

Doing the opposite (small ctau to large ctau) makes the exponent **positive**: $w \propto e^{+(t_1+t_2)/\tau_{eff}}$, so a handful of the rare long-decay events in the input sample get enormous weights and dominate the whole distribution. The result looks right on average but has a huge statistical uncertainty (large effective-variance / few effective entries), and if $\tau_{new} \gg \tau_{old}$ there may be essentially no events left to reweight at all. That is why this exercise uses $10\,\mathrm{m} \to 1\,\mathrm{m}$ and not $1\,\mathrm{m} \to 10\,\mathrm{m}$, and why the acceptance scan always reweights *down* from the next-longer generated sample.

# Ex3 - Calculate the probability of LLP decaying in the muon system

In this exercise you will calculate the probability of the LLP decaying in the muon system by requiring the LLP decay vertex lies within the Muon system.

You will first define the geometric decay region of the muon system based on Fig. 4.1.1 on page 141 of the [Muon Detector Technical Design Report](https://cds.cern.ch/record/343814?ln=en) 

Replace the `REPLACE_ME` in the cell below. 

Then in the last line you will print the probability of LLP decaying in the endcap muon detectors for LLP mean proper decay lenegths (0.1, 1, 10, 100m) that have been generated.

The probability should be on the order of 15-20% for the 1m and 10m sample.

In [ ]:
%%time

for k, T in tree.items():
    
     # define endcap muon system region
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.1
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1050)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 568)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 280)
    
    print(k, np.count_nonzero(np.sum(sel_ms, axis = 1))/len(sel_ms))

### Ex3 (muon system decay region) explanation

We approximate the **endcap muon system** as a shell of two disks at each end of the detector, taking the dimensions from Fig. 4.1.1 (page 141) of the Muon TDR - the same geometry that `util.drawCSCz` and `util.drawCSCsteel` draw in the later notebooks, where the grey bands are the steel absorbers (named `preME11`, `ME11_12`, `ME12_2`, `ME2_3`, `ME3_4`, `beyond`) and the CSC chambers sit in the gaps between them:

| cut | value | where it comes from |
|---|---|---|
| `abs(gLLP_eta) < 2.1` | direction | the CSC endcap stations cover up to about abs(eta) = 2.1 |
| `abs(z) > 568` | cm | ME1/1, the first endcap station, starts at abs(z) = 568 cm (the end of the `preME11` steel) |
| `abs(z) < 1050` | cm | ME4, the last endcap station, ends at abs(z) = 1050 cm (`beyond` = outside CMS starts there) |
| `abs(r) < 280` | cm | radial extent of the endcap disks |

The four conditions are combined with `np.logical_and` to build `sel_ms`, a boolean mask with shape (events, 2 LLPs). `np.sum(sel_ms, axis=1)` counts how many of the two LLPs in an event decay inside that region, and `np.count_nonzero(...)/len(sel_ms)` is the fraction of **events** with at least one such decay: the geometric acceptance.

The vertex branches are in **cm**, so the numbers above are used directly (only the file names are in mm). Expected output: an O(15-20%) probability for the 1 m and 10 m samples, clearly smaller for 0.1 m (the LLP decays much earlier, inside the calorimeters) and for 100 m (most LLPs cross the whole detector without decaying). Other reasonable box choices exist (e.g. a looser box with abs(eta) < 2.4, 400 < abs(z) < 1100 and r < 695 cm, which also accepts barrel-like decays); the physics conclusion does not change.

## calulate acceptance vs LLP mean proper decay length

Here we will extend on what we just did to calculate the acceptance for intermediate $c\tau$ by calling the 
reweighting function that you've written in the previous exercise to calculate the probability for other intermediate proper decay lengths.


You will fill in the definition of `sel_ms`, which is what you defined in the cell above and the arguments for `weight_calc()` to reweight to intermediate lifetimes in the `ctau_list`

The list of acceptances for different `ctau` is then saved in the `acceptance` list

In [ ]:
%%time

ctau_list  = [100, 300, 500, 1000, 2000, 5000, 10000, 50000, 100000, 500000] #mm

acceptance = []
for ct in ctau_list:
    # find which sample to use for reweighting
    ct_source = 0
    for i in OLD_CTAU:
        if ct <= i: 
            ct_source = i
            break
    if ct_source == 0: ct_source = OLD_CTAU[-1]
    print(ct, ct_source) #verify the correct sample is chosen
    
    T = tree['MC_40_{}'.format(ct_source)]
    
    ################## ANSWER #################
    # geometric endcap muon system region, identical to Ex3 (vertices are in cm)
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.1
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1050)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 568)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 280)

    # ctau_list (ct) and ct_source are in mm, gLLP_ctau is in cm -> divide by 10
    weight = weight_calc(np.sum(gLLP_ctau['MC_40_{}'.format(ct_source)], axis = 1), ct/10.0, ct_source/10.0)
    
    #################################################
    
    denominator = np.sum(weight)
    numerator = np.sum(weight[np.sum(sel_ms, axis = 1)>0])
    acceptance.append(numerator/denominator)
    
print(ctau_list, acceptance)

### Acceptance vs. ctau explanation

The loop picks, for every target lifetime `ct` in `ctau_list` (in mm), the *generated* sample whose ctau is the closest one **not smaller** than it (`ct_source`). We then

1. rebuild the same muon-system mask `sel_ms` as in Ex3, and
2. get an event-by-event weight from `weight_calc`, reweighting that sample from `ct_source` to `ct`.

The acceptance is then just the weighted fraction of events with at least one LLP decay in the region:

~~~
denominator = np.sum(weight)                          # all events
numerator   = np.sum(weight[np.sum(sel_ms,axis=1)>0]) # events with a decay in the MS
~~~

Units are the trap here: `ctau_list`/`OLD_CTAU` are in **mm** (they mirror the file names) while `gLLP_ctau` is in **cm**, so both `ct` and `ct_source` are divided by 10 before being handed to `weight_calc`. `np.sum(gLLP_ctau[...], axis=1)` is again $t_1+t_2$.

Because one MC sample is reused for several target lifetimes, a single set of generated samples covers the whole range 0.1 m to 500 m - that is the point of reweighting.

## plot acceptance vs LLP mean proper decay length

Here you will plot a TGraph with `ctau_list` on the x-axis and acceptance on the y-axis.

Documentation for TGraph: https://root.cern.ch/doc/master/classTGraph.html



### Q: Do you understand the shape? 

Play around with the `ctau_list` to add more points to make the plot smoother!

In [ ]:
%%time

c = rt.TCanvas('c','c', 800, 800)
c.SetRightMargin(0.04)

h = rt.TGraph(len(ctau_list),array("d", np.array(ctau_list)/1000.0), array("d", acceptance))

h.GetXaxis().SetTitle('c#tau [m]')
h.GetYaxis().SetTitle('acceptance')
h.GetXaxis().SetLimits(0.1,1000.0)
h.GetYaxis().SetRangeUser(1e-3,2)

h.GetXaxis().SetTitleOffset(1)
h.GetYaxis().SetTitleSize(0.05)
h.GetYaxis().SetTitleOffset(1.5)

h.SetLineWidth(3)
h.SetLineColor(4)
h.SetMarkerColor(4)

h.Draw('LPA')

c.SetLogy()
c.SetLogx()
c.SetTicky(1)
c.SetTickx(1)


tdrstyle.setTDRStyle()
c.Draw()

### Discussion 2.3 - why does the acceptance peak at ctau = 2 m?

The curve has to go to zero at **both** ends:

- very small ctau: the LLP decays promptly, most of the time within the tracker/calorimeters, i.e. **before** the muon system, so it never lands in the decay region;
- very large ctau: the LLP is essentially stable on the scale of CMS and flies out through the muon system **without decaying**;
- in between there is a maximum, where the mean lab-frame decay length $\beta\gamma c\tau$ is comparable to the thickness of the muon-system shell, so a large fraction of the exponential falls inside it.

Why is the peak at ctau ~ 2 m and not at the 6-10 m distance from the interaction point to the muon system? Because the exponential is written in the **lab** frame: the decay length the detector actually sees is $\beta\gamma c\tau$. The LLPs from a 125 GeV Higgs decaying to two 40 GeV LLPs are boosted, with a typical $\beta\gamma$ of a few (roughly 3-5), so a proper ctau of ~2 m corresponds to a mean lab decay length of ~6-10 m, exactly the radius of the muon system. Rewriting the condition as $\beta\gamma c\tau \sim \langle z \rangle$ removes the apparent contradiction; a smaller (larger) boost would move the peak to larger (smaller) ctau.